In [1]:
import os
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import numpy as np
from math import *


In [2]:
def DFT_matrix(N):
    i, j = np.meshgrid(np.arange(N), np.arange(N))
    omega = np.exp( - 2 * pi * 1J / N )
    W = np.power( omega, i * j ) / sqrt(N)
    return np.mat(W)



# 均匀面阵

In [3]:
# 最终修正版函数
def freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt,Nr):
    d=0.5
    Tau = Nc
    tau = np.random.rand(1,N)*Tau

    if isinstance(Nt, list):
        N_t = Nt[0]*Nt[1]
    else:
        N_t = Nt
    N_r = Nr

    fait = np.random.rand(1,N)*2*pi
    fair = np.random.rand(1,N)*2*pi
    theatt = np.random.rand(1,N)*2*pi
    theatr = np.random.rand(1,N)*2*pi

    # --- 核心修改：将 np.mat 替换为 np.array ---
    A_t = np.zeros([N_t,N], dtype=complex)
    A_r = np.zeros([N_r,N], dtype=complex)
    alpha = np.zeros([N], dtype=complex) # np.squeeze 不再需要

    H_f = np.zeros([N_r,Nc,N_t], dtype=complex)

    for i in range(N):
        if isinstance(Nt, list):
            # --- 核心修改：确保所有中间变量都是 ndarray ---
            n_t1 = np.arange(Nt[0]).reshape(Nt[0],1)
            n_t2 = np.arange(Nt[1]).reshape(Nt[1],1)
            at1 = np.exp(-2j*pi*d*n_t1*np.cos(fait[0,i])*np.sin(theatt[0,i]))
            at2 = np.exp(-2j*pi*d*n_t2*np.sin(fait[0,i]))
            A_t[:,i] = np.kron(at1,at2).flatten() # 使用 .flatten() 确保维度正确
        else:
            n_t = np.arange(N_t).reshape(N_t,1)
            A_t[:,i] = np.exp(-2j*pi*d*n_t*np.sin(fait[0,i])).flatten()

        n_r = np.arange(N_r).reshape(N_r,1)
        A_r[:,i] = np.exp(-2j*pi*d*n_r*np.sin(fair[0,i])).flatten()

        aa = (np.random.randn(1,1)+1j*np.random.randn(1,1))*np.sqrt(sigma_2_alpha/2/N)
        alpha[i] = aa[0,0]

    for k in range(Nc):
        P = np.diag(alpha*np.exp(-1j*2*pi*tau[0,:]*k/Nc)) # tau 需要索引

        # --- 核心修改：A_t.T.conj() 代替 A_t.H ---
        # 对于 ndarray, .H 代表厄米特转置（与 .T.conj() 等价）
        H_f[:,k,:] = A_r @ P @ A_t.T.conj() # 使用 @ 运算符进行矩阵乘法，更现代

    return H_f, A_t, alpha

In [4]:
Nc = 16
sigma_2_alpha = 1
Nt = [12,12]
N_t = Nt[0]*Nt[1]
Nr = 16

B = 30
D = 1000; #角度采样点数
L = 8

SNR_dB = 5
K = 4
snr =  10**(SNR_dB/10)/K

N_BATCH_train = 40
N_BATCH_test  = 8
BATCH_SIZE = 128
N_H_train = N_BATCH_train*BATCH_SIZE
N_H_test = N_BATCH_test*BATCH_SIZE

os.makedirs('data2', exist_ok=True)

In [5]:

for N in range(2,3):
    H_torch = torch.zeros([BATCH_SIZE*N_BATCH_train, K, Nr, Nc, N_t*2])
    for i in range(N_BATCH_train):
        H = np.zeros([BATCH_SIZE,K,Nr,Nc,N_t],dtype=complex)  #第0个维度是样本 第1个维度是用户，第2个维度是子载波，第3个维度是天线
        for j in range(BATCH_SIZE):
            for k in range(K):
                H_f, A_t, alpha = freqency_sparse_SV_channel0(Nc, N, sigma_2_alpha, Nt, Nr)
                H[j, k, :, :, :] = H_f
    
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE), :, :, :, 0:N_t] = torch.from_numpy(np.real(H))
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE), :, :, :, N_t:2*N_t] = torch.from_numpy(np.imag(H))
        print(i)
    torch.save(H_torch,'data2/H_train_UPA'+str(N)+'Lp_1.pt')
    
    H_torch = torch.zeros([BATCH_SIZE*N_BATCH_train,K,Nr,Nc,N_t*2])
    for i in range(N_BATCH_train):
        H = np.zeros([BATCH_SIZE,K,Nr,Nc,N_t],dtype=complex)  #第0个维度是样本 第1个维度是用户，第2个维度是子载波，第3个维度是天线
        for j in range(BATCH_SIZE):
            for k in range(K):
                # 在调用时传入 Nr
                H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt,Nr)
                H[j,k,:,:,:] = H_f
    
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,:,0:N_t] = torch.from_numpy(np.real(H))
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
        print(i)
    torch.save(H_torch,'data2/H_train_UPA'+str(N)+'Lp_2.pt')

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39


In [6]:
for N in range(2,3):
    H_torch = torch.zeros([BATCH_SIZE*N_BATCH_test, K, Nr, Nc, N_t*2])
    for i in range(N_BATCH_test):
        H = np.zeros([BATCH_SIZE, K, Nr, Nc, N_t], dtype=complex) #第0个维度是样本 第1个维度是用户，第2个维度是子载波，第3个维度是天线
        for j in range(BATCH_SIZE):
            for k in range(K):
                # 在调用时传入 Nr
                H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt,Nr)
                H[j, k, :, :, :] = H_f
    
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE), :, :, :, 0:N_t] = torch.from_numpy(np.real(H))
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE), :, :, :, N_t:2*N_t] = torch.from_numpy(np.imag(H))
        print(i)
    torch.save(H_torch,'data2/H_test_UPA'+str(N)+'Lp.pt')

0
1
2
3
4
5
6
7


In [7]:
# import scipy.io as io
#
# H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #第0个维度是样本 第1个维度是用户，第2个维度是子载波，第3个维度是天线
# for j in range(BATCH_SIZE):
#     for k in range(K):
#         H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
#         H[j,k,:,:] = H_f
# H_torch = torch.zeros([BATCH_SIZE,K,Nc,N_t*2])
# H_torch[:,:,:,0:N_t] = torch.from_numpy(np.real(H))
# H_torch[:,:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
# #torch.save(H_torch,'data/H_test_UPA.pt')
# print(Nt)
#
#
# dataNew = os.path.join('data2', 'H_UPA.mat')
# io.savemat(dataNew, {'H_UPA': H})